## Data Ingestion Using Web Scraping

When a site offers no API, we can extract data directly from its HTML using
`requests` and BeautifulSoup. Always respect a site's `robots.txt` and terms of
service before scraping.

In [1]:
# Data Ingestion using Web Scraping.
# When a website offers no API, we can extract data directly from its HTML. Here
# we fetch a page from books.toscrape.com, parse the HTML with BeautifulSoup,
# and pull out each book's title, price, image URL and availability. The values
# are collected into lists and finally assembled into a Pandas DataFrame.
# (Always check a site's robots.txt and terms of service before scraping.)

import requests
from bs4 import BeautifulSoup
import pandas as pd

# the web page we want to scrape
url = "https://books.toscrape.com/catalogue/page-1.html"

# send an HTTP GET request to download the page's HTML
response = requests.get(url)

# stop early if the page could not be fetched (200 means success)
if response.status_code != 200 :
      print("Failed to retrieve the webpage.")
      exit( )

# parse the raw HTML text into a navigable BeautifulSoup object
soup = BeautifulSoup(response.text, 'html.parser')

# every book on the page sits inside an <article class="product_pod"> tag;
# find_all( ) returns a list of all such book containers
books = soup.find_all('article', class_='product_pod')

# empty lists to collect one field per book
titles, prices, images, availabilities = [ ], [ ], [ ], [ ]

# walk through each book container and pull out the fields we need
for book in books :
      # title is stored in the 'title' attribute of the <a> inside the <h3>
      title = book.h3.a['title']

      # price text lives in a <p class="price_color">
      price = book.find('p', class_='price_color').text

      # the <img> src is a relative path, so prepend the base URL to make it absolute
      image = "https://books.toscrape.com/" + book.find('img')['src'].replace('../', '')

      # availability text is in a <p class="instock availability">; strip( ) trims whitespace
      availability = book.find('p', class_='instock availability').text.strip( )

      # add this book's values to their respective lists
      titles.append(title)
      prices.append(price)
      images.append(image)
      availabilities.append(availability)

# combine the four parallel lists into a single DataFrame, one column each
books_df = pd.DataFrame({
      'Title': titles,
      'Price': prices,
      'Image URL': images,
      'Availability': availabilities })
print(books_df)


                                                Title    Price  \
0                                A Light in the Attic  Â£51.77   
1                                  Tipping the Velvet  Â£53.74   
2                                          Soumission  Â£50.10   
3                                       Sharp Objects  Â£47.82   
4               Sapiens: A Brief History of Humankind  Â£54.23   
5                                     The Requiem Red  Â£22.65   
6   The Dirty Little Secrets of Getting Your Dream...  Â£33.34   
7   The Coming Woman: A Novel Based on the Life of...  Â£17.93   
8   The Boys in the Boat: Nine Americans and Their...  Â£22.60   
9                                     The Black Maria  Â£52.15   
10     Starving Hearts (Triangular Trade Trilogy, #1)  Â£13.99   
11                              Shakespeare's Sonnets  Â£20.66   
12                                        Set Me Free  Â£17.46   
13  Scott Pilgrim's Precious Little Life (Scott Pi...  Â£52.29   
14        